# 04 — KNN Model Training

Assembles the full feature matrix, prunes sparse columns, trains a cosine-similarity K-Nearest Neighbours model, and saves all artefacts needed by the app.

**What it does:**
- Loads and stacks all feature blocks into a single matrix (same logic as notebook 03)
- Prunes columns below a safe sparsity threshold so no album with features loses all its signal
- Subsets to albums that have at least one feature — albums with no signal are excluded from the fitted model
- L2-normalises the matrix so cosine similarity reduces to a dot product at query time
- Fits a brute-force `NearestNeighbors` model with cosine distance using scikit-learn
- Saves the trained model, normalised matrix, and album ID index to `data/model/`

**Inputs:** `data/features/album_ids.pkl`, `data/features/album_tags_matrix.npz`, `data/features/album_labels_matrix.npz`, `data/features/album_types_matrix.npz`, `data/features/album_ratings_matrix.npz`, `data/mb_album.parquet`

**Outputs to `data/model/`:** `knn_model.joblib`, `X_knn_norm.npz`, `album_ids_annotated.npy`, `has_features.npy`

**Run after:** `02-feature-ratings.ipynb` | **Run before:** `05-knn-query.ipynb`

## Imports

Standard scientific Python stack plus `scipy.sparse` for the sparse feature matrices built in earlier notebooks. `os.makedirs` is called defensively here so the notebook is safe to run even if the `data/features/` directory does not yet exist.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, load_npz

os.makedirs('../data/features', exist_ok=True)

## Load feature blocks, expand to full album universe, and assemble final matrix

This is identical to the matrix-assembly logic in notebook 03. The four sparse feature blocks (tags, labels, types, ratings) were each built against the subset of albums that had data, so their row counts may be smaller than the total MusicBrainz album universe. The `_expand` helper re-indexes each block into the full `n_full × columns` shape so all four blocks share the same row order before being joined with `hstack`.

`hstack` stacks the blocks side-by-side, producing one wide sparse matrix where each row is an album and each column is a feature from one of the four sources. The result is stored as CSR (Compressed Sparse Row) format, which is efficient for row-slicing operations used later.

In [ ]:
# Load row index
with open('../data/features/album_ids.pkl', 'rb') as f:
    album_id_order = pickle.load(f)

# Load feature blocks
X_tags    = load_npz('../data/features/album_tags_matrix.npz')
X_labels  = load_npz('../data/features/album_labels_matrix.npz')
X_types   = load_npz('../data/features/album_types_matrix.npz')
X_ratings = load_npz('../data/features/album_ratings_matrix.npz')

# Expand to full album universe
full_album_ids = pd.Index(
    pd.read_parquet('../data/mb_album.parquet', columns=['id'])['id'].sort_values()
)

if len(album_id_order) < len(full_album_ids):
    print(f"Expanding matrices from {len(album_id_order):,} → {len(full_album_ids):,} albums...")
    current_pos = full_album_ids.get_indexer(album_id_order)
    n_full = len(full_album_ids)

    def _expand(X, row_pos, n):
        coo = X.tocoo()
        return csr_matrix((coo.data, (row_pos[coo.row], coo.col)), shape=(n, X.shape[1]))

    X_tags    = _expand(X_tags,    current_pos, n_full)
    X_labels  = _expand(X_labels,  current_pos, n_full)
    X_types   = _expand(X_types,   current_pos, n_full)
    X_ratings = _expand(X_ratings, current_pos, n_full)
    album_id_order = full_album_ids.tolist()

# Assemble final matrix
X_final_album_knn = hstack([X_tags, X_labels, X_types, X_ratings]).tocsr()

print(f"X_final_album_knn: {X_final_album_knn.shape[0]:,} albums x {X_final_album_knn.shape[1]:,} features  (nnz={X_final_album_knn.nnz:,})")
print(f"album_id_order length: {len(album_id_order):,}")

## Per-column non-zero statistics

Before pruning anything, we need to understand how sparse the columns are. `col_nnz` is computed from the CSC (Compressed Sparse Column) format because CSC stores `indptr` per column — `np.diff(indptr)` gives the count of stored entries in each column in O(columns) time without iterating.

`col_density` (fraction of albums with a non-zero value for that column) is a more intuitive measure than raw counts. The percentile breakdown reveals the power-law distribution typical of user-generated tags: a small number of columns (popular tags, common genres) cover many albums, while the long tail covers very few.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# --- per-column stats across the full matrix ---
col_nnz   = np.diff(X_final_album_knn.tocsc().indptr)   # non-zeros per column
col_sums  = np.asarray(X_final_album_knn.sum(axis=0)).ravel()
n_albums  = X_final_album_knn.shape[0]
col_density = col_nnz / n_albums  # fraction of albums with a non-zero value

print(f"Total columns : {len(col_nnz):,}")
print(f"Zero columns  : {(col_nnz == 0).sum():,}")
print(f"Singleton cols: {(col_nnz == 1).sum():,}  (exactly 1 album)")
print(f"Columns with ≤5 albums  : {(col_nnz <= 5).sum():,}")
print(f"Columns with ≤10 albums : {(col_nnz <= 10).sum():,}")
print(f"\ncol_nnz percentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(f"  p{p:>2}: {np.percentile(col_nnz, p):.0f}")

## Column nnz distribution plots

Two views of the same data. The linear histogram makes it easy to see how many columns have very low coverage. The log-log histogram reveals the full shape of the distribution including the rare columns that cover thousands of albums. Visualising both scales is important here because a linear plot would compress the long tail into near-zero and hide the structure that drives threshold selection.

In [ ]:
# --- distribution of column nnz (log scale) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(col_nnz, bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Albums per column (nnz)')
axes[0].set_ylabel('Number of columns')
axes[0].set_title('Column nnz distribution (linear)')

axes[1].hist(col_nnz[col_nnz > 0], bins=100, color='steelblue', edgecolor='none', log=True)
axes[1].set_xscale('log')
axes[1].set_xlabel('Albums per column (nnz, log scale)')
axes[1].set_ylabel('Number of columns (log scale)')
axes[1].set_title('Column nnz distribution (log-log)')

plt.tight_layout()
plt.show()

## Per-block column breakdown

The four feature blocks (tags, labels, types, ratings) have very different column counts and sparsity profiles. This breakdown is useful when deciding thresholds: a `≤5 nnz` column in `X_ratings` (a block with relatively few, high-confidence values) means something different from a `≤5 nnz` column in `X_tags` (where tags are user-contributed and very long-tailed). Reviewing each block separately prevents a single dominant block from masking problems in the others.

In [ ]:
# --- per-block breakdown ---
block_sizes = {
    'X_tags':    X_tags.shape[1],
    'X_labels':  X_labels.shape[1],
    'X_types':   X_types.shape[1],
    'X_ratings': X_ratings.shape[1],
}

block_cols = {}
start = 0
for name, size in block_sizes.items():
    block_cols[name] = col_nnz[start:start + size]
    start += size

print(f"{'Block':<12} {'cols':>6}  {'zero':>6}  {'≤5 nnz':>7}  {'median nnz':>10}  {'max nnz':>10}")
print("-" * 60)
for name, nnz in block_cols.items():
    print(f"{name:<12} {len(nnz):>6,}  {(nnz==0).sum():>6,}  {(nnz<=5).sum():>7,}  "
          f"{np.median(nnz):>10.0f}  {nnz.max():>10,}")

## CDF analysis — columns retained vs. min-nnz cut

The CDF plot shows how aggressively a given minimum-nnz threshold prunes the column space. The two reference lines at `nnz=10` and `nnz=50` bracket a reasonable operating range: below 10, many near-useless singleton and doubleton columns remain; above 50, we risk dropping columns that carry real signal for niche genres. The table above the plot gives exact numbers for common thresholds so the eventual choice in the next cell can be justified precisely.

In [ ]:
# --- column density CDF — shows what threshold captures what share of columns ---
thresholds = [1, 2, 5, 10, 25, 50, 100, 250, 500]
print("Min-nnz threshold  |  columns kept  |  % of total")
print("-" * 50)
for t in thresholds:
    kept = (col_nnz >= t).sum()
    print(f"  ≥ {t:<4}            |  {kept:>6,}         |  {kept/len(col_nnz)*100:.1f}%")

fig, ax = plt.subplots(figsize=(9, 4))
sorted_nnz = np.sort(col_nnz)
ax.plot(sorted_nnz, np.linspace(0, 1, len(sorted_nnz)), color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('Minimum nnz threshold (log scale)')
ax.set_ylabel('Fraction of columns retained')
ax.set_title('CDF: columns retained vs. min-nnz cut')
ax.axvline(10,  color='orange', linestyle='--', label='nnz=10')
ax.axvline(50,  color='red',    linestyle='--', label='nnz=50')
ax.legend()
plt.tight_layout()
plt.show()

## Album zeroing analysis — impact of column pruning on albums

Column pruning has a side-effect: if all of an album's non-zero columns are below the threshold, that album becomes an all-zero row after pruning. An all-zero row cannot produce a meaningful cosine similarity — its "direction" is undefined — so it would silently return garbage neighbours. This cell measures exactly that risk.

`max_col_nnz_per_album` stores, for each album, the highest column frequency among all of its non-zero entries. If this maximum is below the threshold, every column the album touches will be pruned. The `np.maximum.reduceat` trick efficiently computes the per-row maximum directly from the CSR data and `indptr` arrays without building a second matrix. The table lets us pick a threshold knowing precisely how many albums would be silently zeroed.

In [ ]:
# For each album, find the highest col_nnz among its non-zero columns.
# If that max < threshold t, the album becomes a zero row after pruning.
# Uses np.maximum.reduceat to avoid building a second sparse matrix.
col_nnz_vals = col_nnz[X_final_album_knn.indices]  # col_nnz for every stored entry

row_lengths = np.diff(X_final_album_knn.indptr)
nonempty_rows = np.where(row_lengths > 0)[0]
row_starts = X_final_album_knn.indptr[nonempty_rows]

max_col_nnz_per_album = np.zeros(n_albums, dtype=col_nnz.dtype)
max_col_nnz_per_album[nonempty_rows] = np.maximum.reduceat(col_nnz_vals, row_starts)

already_empty = (max_col_nnz_per_album == 0).sum()
print(f"Albums already with no features: {already_empty:,}  ({already_empty/n_albums*100:.2f}%)\n")

thresholds = [1, 2, 5, 10, 25, 50, 100, 250, 500]
print(f"{'threshold':>10} | {'cols kept':>10} | {'cols %':>7} | {'albums zeroed':>14} | {'albums %':>9} | {'newly zeroed':>13}")
print("-" * 75)
for t in thresholds:
    kept_cols = (col_nnz >= t).sum()
    zeroed    = (max_col_nnz_per_album < t).sum()
    newly     = zeroed - already_empty
    print(f"{t:>10} | {kept_cols:>10,} | {kept_cols/len(col_nnz)*100:>6.1f}% | "
          f"{zeroed:>14,} | {zeroed/n_albums*100:>8.3f}% | {newly:>13,}")

## has_features mask — identify albums with any feature signal

`has_features` is a boolean array aligned to the full album universe. An album is `True` if it has at least one non-zero entry in the pre-pruning matrix. Albums that are `False` here have absolutely no feature data from any of the four blocks — no tags, no genre labels, no release type signal, and no ratings. They cannot contribute to or receive a meaningful recommendation.

This mask is saved as a model artefact so the app can quickly gate whether a user-selected album is even queryable, before attempting an expensive model lookup.

In [ ]:
# Boolean mask: which albums have at least one feature
has_features = row_lengths > 0

print(f"Albums with features    : {has_features.sum():>10,}  ({has_features.mean()*100:.1f}%)")
print(f"Albums without features : {(~has_features).sum():>10,}  ({(~has_features).mean()*100:.1f}%)")
print(f"Total                   : {len(has_features):>10,}")
print()
print("At query time:")
print("  - If the query album has no features → return 'no recommendations available'")
print("  - Only albums with features can be used as query seeds")

## Safe threshold and column pruning

Rather than picking a threshold by hand from the CDF analysis, the safe threshold is derived automatically: it is the minimum of `max_col_nnz_per_album` across all albums that have features. This is the highest threshold we can apply without zeroing a single album that currently has at least one feature. It is a principled lower bound — any threshold above this value would silently destroy signal for at least one album.

The column boolean mask `keep_cols` is then applied to `X_final_album_knn` to produce the pruned matrix `X_knn`. This reduces noise from ultra-rare tags and speeds up subsequent operations.

In [ ]:
# Find the highest threshold that zeroes no additional albums.
# = the minimum "best column nnz" across all albums that have features.
safe_threshold = int(max_col_nnz_per_album[has_features].min())

keep_cols = col_nnz >= safe_threshold
X_knn = X_final_album_knn[:, keep_cols]

print(f"Safe threshold : {safe_threshold} (no album with features loses all its features)")
print(f"Columns before : {X_final_album_knn.shape[1]:,}")
print(f"Columns after  : {X_knn.shape[1]:,}  ({keep_cols.sum()/len(col_nnz)*100:.1f}% retained)")
print(f"nnz before     : {X_final_album_knn.nnz:,}")
print(f"nnz after      : {X_knn.nnz:,}")

## Subset to albums with features, remove NaNs, and L2-normalise

Three distinct steps happen here:

**1. Subset to `has_features` rows.** The matrix used for fitting contains only albums that have at least one non-zero feature. Zero rows are excluded for two reasons: (a) a zero vector has no defined direction, so cosine distance to it is undefined; (b) brute-force search computes distances to every row in the index — including zero rows wastes compute and could produce misleadingly close distances for other sparse albums.

**2. NaN removal.** Sparse matrices store only explicitly set values, but floating-point NaNs can creep in during feature computation (e.g. a division by zero when normalising an empty rating histogram). If a NaN is stored, `normalize` propagates it across the whole row, silently corrupting the model. `np.nan_to_num` replaces NaNs with zero in-place, then `eliminate_zeros` removes those entries from the sparse structure.

**3. L2 normalisation.** After normalisation every row has unit length: `‖x‖₂ = 1`. For two unit vectors, `cosine_distance = 1 − (x · y)`, so cosine similarity is exactly the dot product. scikit-learn's `NearestNeighbors` with `metric='cosine'` computes this correctly regardless, but normalising beforehand means the stored vectors are ready for direct dot-product queries if the model is ever swapped out for an approximate search library (e.g. FAISS, Annoy) that works natively with inner products.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize

# Subset to albums with features only — zero rows add no signal and slow down brute-force search
X_knn_annotated    = X_knn[has_features].copy()
album_ids_annotated = np.array(album_id_order)[has_features]

# Remove any NaN values stored explicitly in the sparse data array
nan_count = np.isnan(X_knn_annotated.data).sum()
if nan_count:
    print(f"Removing {nan_count:,} NaN entries from sparse data...")
    np.nan_to_num(X_knn_annotated.data, nan=0.0, copy=False)
    X_knn_annotated.eliminate_zeros()

# L2-normalise so cosine similarity reduces to a dot product (faster at query time)
X_knn_norm = normalize(X_knn_annotated, norm='l2')

print(f"Fitting on {X_knn_norm.shape[0]:,} albums x {X_knn_norm.shape[1]:,} features")

## Fit the NearestNeighbors model

`algorithm='brute'` means the model stores every row and computes the exact distance to all of them at query time — there is no index structure. Brute-force is the correct choice here because:

- The dataset is on the order of tens of thousands of albums, not millions, so an exact scan is fast enough in practice.
- Approximate methods (e.g. ball-tree, k-d tree) only help for Euclidean distance in low dimensions; they do not improve cosine search on high-dimensional sparse data.
- Exact results are important during prototyping: approximations would introduce noise that makes it harder to evaluate whether the features themselves are working.

`n_jobs=-1` parallelises distance computation across all available CPU cores at query time.

In [ ]:
model = NearestNeighbors(metric='cosine', algorithm='brute', n_jobs=-1)
model.fit(X_knn_norm)
print("Model fitted.")

## Sanity check — query the first album, print 10 nearest neighbours

A minimal smoke test to verify the model is working before saving. `n_neighbors=11` is requested so that after excluding `rank=0` (the query album itself, which always returns at distance ≈ 0.0) we still get 10 results. If the distances are all identical or the query album does not appear at rank 0, something has gone wrong with the matrix assembly or normalisation.

In [ ]:
# Sanity check — query the first annotated album and print its 10 nearest neighbours
distances, indices = model.kneighbors(X_knn_norm[0], n_neighbors=11)

print(f"Query album id : {album_ids_annotated[0]}")
print(f"\n{'rank':<6} {'album_id':<40} {'cosine distance':>15}")
print("-" * 62)
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
    label = "(query)" if rank == 0 else ""
    print(f"{rank:<6} {str(album_ids_annotated[idx]):<40} {dist:>15.4f}  {label}")

## Save model artefacts

Four files are written to `data/model/`. Each serves a specific purpose downstream:

- **`knn_model.joblib`** — the fitted `NearestNeighbors` object. `joblib` is preferred over `pickle` for scikit-learn models because it handles numpy arrays and sparse matrices more efficiently. The app loads this to call `model.kneighbors()` at query time.
- **`X_knn_norm.npz`** — the L2-normalised sparse feature matrix, one row per recommendable album. Saved separately from the model so it can be sliced by row index to retrieve the query vector without going through the model object.
- **`album_ids_annotated.npy`** — a 1-D array mapping each row index in `X_knn_norm` back to a MusicBrainz album UUID. This is the bridge between the model's integer row indices and the album IDs used everywhere else in the system.
- **`has_features.npy`** — the full-universe boolean mask. The app uses this to filter the album selectbox so users can only query albums that are actually in the model index.

In [ ]:
import joblib
from scipy.sparse import save_npz

os.makedirs('../data/model', exist_ok=True)

joblib.dump(model,               '../data/model/knn_model.joblib')
save_npz('../data/model/X_knn_norm.npz', X_knn_norm)
np.save('../data/model/album_ids_annotated.npy', album_ids_annotated)
np.save('../data/model/has_features.npy',        has_features)

print("Saved:"             )
print("  ../data/model/knn_model.joblib")
print("  ../data/model/X_knn_norm.npz")
print("  ../data/model/album_ids_annotated.npy")
print("  ../data/model/has_features.npy")